# Voice activity detection (VAD): hands-on

Companion to the manual's VAD section. You will:

1. load 60 s of a Thai meeting, recorded twice: on a close-talk microphone and on a microphone 3 m away,
2. run Silero, TEN VAD and WebRTC on it and see where each finds speech,
3. watch Silero miss the distant recording, then fix it by raising the level,
4. score every run against a human reference, and cut the speech into WAV files for speech-to-text.

Audio: [LOTUSDIS](https://huggingface.co/datasets/nectec/LOTUSDIS) (NECTEC), CC BY-SA 4.0. See the README for the exact source.

## Setup

Colab already has PyTorch, NumPy, pandas and Matplotlib. This installs the three VAD libraries at the versions the manual was tested with.

In [ ]:
!pip install -q silero-vad==6.2.2 onnxruntime ten-vad==1.0.6.9 webrtcvad-wheels

Get the sample files. Run locally from a clone, they are already here. In Colab, set `REPO_RAW` to the repository's raw URL, or upload the `data/` folder with the Files panel on the left.

In [ ]:
from pathlib import Path
import urllib.request

REPO_RAW = ""  # e.g. "https://raw.githubusercontent.com/<owner>/thai-speech-notebooks/main/"
DATA = Path("data/lotusdis-S002_T062")
for name in ["close.wav", "distant.wav", "reference.csv"]:
    path = DATA / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + str(path), path)
print(sorted(p.name for p in DATA.iterdir()))

## 1. Check the audio format

All three models want 16 kHz, mono, 16-bit WAV. The sample files already are. For your own audio, convert first:

```bash
ffmpeg -i input.mp3 -ac 1 -ar 16000 audio_16k.wav
```

In [ ]:
import wave
import numpy as np
import pandas as pd
from IPython.display import Audio, display

def read(path):
    """Sample rate and 16-bit samples of a mono WAV."""
    with wave.open(str(path), "rb") as f:
        assert f.getnchannels() == 1 and f.getsampwidth() == 2, "convert to 16-bit mono first"
        return f.getframerate(), np.frombuffer(f.readframes(f.getnframes()), dtype=np.int16)

sr, close = read(DATA / "close.wav")
_, distant = read(DATA / "distant.wav")
reference = list(pd.read_csv(DATA / "reference.csv").itertuples(index=False, name=None))
print(f"{sr} Hz, {len(close) / sr:.0f} s; reference has {len(reference)} speech segments")

Listen to both. Same people, same moment: the distant one is much quieter and more muffled.

In [ ]:
display(Audio(close, rate=sr))
display(Audio(distant, rate=sr))

## 2. Run the three models

Each function returns the speech segments as `(start, end)` pairs in seconds, plus the per-frame score for plotting. They follow the manual's Getting started snippets: Silero with its library defaults (threshold 0.5), TEN VAD at threshold 0.5, WebRTC at mode 3 with 30 ms frames. TEN VAD and WebRTC have no smoothing, so their segments come straight from the frame flags.

In [ ]:
import torch
import webrtcvad
from silero_vad import load_silero_vad, get_speech_timestamps
from ten_vad import TenVad

SILERO = load_silero_vad()

def runs(flags, hop):
    """(start, end) of each run of True frames, in seconds."""
    d = np.diff(np.r_[0, np.asarray(flags, int), 0])
    return [(a * hop, b * hop) for a, b in zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1))]

def silero(pcm, threshold=0.5):
    wav = torch.from_numpy(pcm / 2**15).float()
    segments = get_speech_timestamps(wav, SILERO, sampling_rate=sr, threshold=threshold, return_seconds=True)
    probs = [SILERO(wav[i:i + 512], sr).item() for i in range(0, len(wav) - 511, 512)]
    SILERO.reset_states()
    return [(s["start"], s["end"]) for s in segments], np.arange(len(probs)) * 512 / sr, np.array(probs)

def ten(pcm, threshold=0.5, hop=256):
    vad = TenVad(hop, threshold)
    probs, flags = zip(*(vad.process(pcm[i:i + hop]) for i in range(0, len(pcm) - hop + 1, hop)))
    return runs(flags, hop / sr), np.arange(len(probs)) * hop / sr, np.array(probs)

def webrtc(pcm, mode=3, frame_ms=30):
    vad, n = webrtcvad.Vad(mode), sr * frame_ms // 1000
    flags = [vad.is_speech(pcm[i:i + n].tobytes(), sr) for i in range(0, len(pcm) - n + 1, n)]
    return runs(flags, n / sr), np.arange(len(flags)) * n / sr, np.array(flags, float)

MODELS = {"Silero": silero, "TEN VAD": ten, "WebRTC": webrtc}

A plot per model: grey is speech in the human reference, the line is the model's score, the blue bar under it is what the model called speech.

In [ ]:
import matplotlib.pyplot as plt

def plot(pcm, title):
    fig, axes = plt.subplots(len(MODELS), 1, figsize=(12, 1.8 * len(MODELS)), sharex=True)
    for ax, (name, run) in zip(axes, MODELS.items()):
        segments, t, score = run(pcm)
        for s, e in reference:
            ax.axvspan(s, e, color="0.85", lw=0)
        ax.plot(t, score, lw=0.8, color="k")
        for s, e in segments:
            ax.plot([s, e], [-0.12, -0.12], lw=5, color="tab:blue", solid_capstyle="butt")
        ax.axhline(0.5, ls="--", lw=0.8, color="tab:orange")
        ax.set_ylim(-0.2, 1.05)
        ax.set_ylabel(name)
    axes[0].set_title(title)
    axes[-1].set_xlabel("seconds")
    plt.tight_layout()
    plt.show()

plot(close, "Close-talk microphone")

## 3. The same speech, 3 m away

Silero is the manual's default, but on a distant microphone it misses most of the speech. Watch its line stay near 0 while TEN VAD still follows the speech.

In [ ]:
plot(distant, "Distant microphone (3 m)")

## 4. Raise the level, then try again

The level is the root mean square (RMS) of the samples, in dBFS: 0 dBFS is the loudest a sample can be. The manual raises distant audio to −20 dBFS. Gain G = target − level, and every sample is multiplied by 10^(G/20). Peaks pushed past 0 dBFS are cut off (clipping): this prints how much.

This needs the whole recording first, so it cannot run on live audio. That is why the manual recommends TEN VAD for live distant-mic audio.

In [ ]:
def level_dbfs(pcm):
    x = pcm / 2**15
    return 20 * np.log10(np.sqrt(np.mean(x**2)))

level = level_dbfs(distant)
gain = -20 - level
scaled = distant * 10 ** (gain / 20)
louder = np.clip(np.round(scaled), -32768, 32767).astype(np.int16)
print(f"level {level:.1f} dBFS, gain {gain:+.1f} dB (every sample x {10 ** (gain / 20):.2f})")
print(f"clipped samples: {np.mean(np.abs(scaled) > 32767):.3%}")
display(Audio(louder, rate=sr))

In [ ]:
plot(louder, f"Distant microphone after {gain:+.1f} dB")

## 5. Score against the reference

FN rate: share of reference speech the model missed. FP rate: share of non-speech it called speech. DCF = 0.75 × FN rate + 0.25 × FP rate, so a missed second costs 3 times a false one; lower is better.

This is a rough, frame-by-frame score on 60 s. The manual's numbers use pyannote.metrics with a 200 ms collar on longer recordings, so expect them to differ.

In [ ]:
def score(segments, duration, step=0.01):
    t = np.arange(0, duration, step)
    inside = lambda segs: np.any([(t >= s) & (t < e) for s, e in segs], axis=0) if segs else np.zeros(len(t), bool)
    called, truth = inside(segments), inside(reference)
    fn = (~called & truth).sum() / truth.sum()
    fp = (called & ~truth).sum() / (~truth).sum()
    return fn, fp, 0.75 * fn + 0.25 * fp

rows = []
for audio_name, pcm in {"close": close, "distant": distant, "distant + gain": louder}.items():
    for name, run in MODELS.items():
        fn, fp, dcf = score(run(pcm)[0], len(pcm) / sr)
        rows.append({"audio": audio_name, "model": name, "FN rate": fn, "FP rate": fp, "DCF": dcf})
pd.DataFrame(rows).pivot(index="model", columns="audio", values="DCF").round(3)

## 6. Check by ear, then save segments for speech-to-text

Scores come from one reference. Before trusting a model on your own audio, listen: is each segment speech, starting and ending with the words? Here are three segments Silero found on the louder audio.

In [ ]:
segments = silero(louder)[0]
for s, e in segments[:3]:
    print(f"{s:.2f}s -> {e:.2f}s")
    display(Audio(louder[int(s * sr):int(e * sr)], rate=sr))

Write each segment to its own WAV, the input the speech-to-text section starts from.

In [ ]:
out = Path("segments")
out.mkdir(exist_ok=True)
for i, (s, e) in enumerate(segments):
    with wave.open(str(out / f"segment_{i:04d}.wav"), "wb") as f:
        f.setnchannels(1)
        f.setsampwidth(2)
        f.setframerate(sr)
        f.writeframes(louder[int(s * sr):int(e * sr)].tobytes())
print(len(segments), "segments written to", out)

## Try next

- Lower Silero's threshold on the distant audio: `plot` uses 0.5, try `silero(distant, threshold=0.1)`. Does any threshold fix it without the gain?
- WebRTC on the distant audio at mode 0 instead of 3: `webrtc(distant, mode=0)`.
- Run your own recording: convert it with the ffmpeg command in step 1, `read()` it, and `plot()` it (the grey reference shading only fits the sample clip).